<a href="https://colab.research.google.com/github/RudreshKj19/CODSOFT/blob/main/model_evaluation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 📦 Import Libraries

In [ ]:
import os
import gc
import pickle
import numpy as np
import cv2
from PIL import Image
import matplotlib.pyplot as plt
import seaborn as sns

import tensorflow as tf
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (Conv2D, MaxPooling2D, Flatten,
                                     Dense, Dropout, BatchNormalization)
from tensorflow.keras.preprocessing.image import ImageDataGenerator

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.decomposition import PCA
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

print("✅ All libraries imported successfully.")

In [ ]:

dataset_path = r"D:\VS Code\Python\plant disease idetification\plantvillage dataset\color"

svm_pred     = svm_model.predict(X_test_pca)
svm_accuracy = accuracy_score(y_test_s, svm_pred)

rf_pred      = rf_model.predict(X_test_pca)
rf_accuracy  = accuracy_score(y_test_s, rf_pred)

cnn_loss, cnn_accuracy = model.evaluate(val_gen, verbose=0)

y_pred_cnn, y_true_cnn = [], []
val_gen.reset()
for imgs, lbls in val_gen:
    preds = model.predict(imgs, verbose=0)
    y_pred_cnn.extend(np.argmax(preds, axis=1))
    y_true_cnn.extend(np.argmax(lbls,  axis=1))
    if len(y_true_cnn) >= val_gen.n:
        break
y_pred_cnn = np.array(y_pred_cnn[:val_gen.n])
y_true_cnn = np.array(y_true_cnn[:val_gen.n])

print(f"SVM Accuracy           : {svm_accuracy*100:.2f}%  (on 5000-sample subset)")
print(f"Random Forest Accuracy : {rf_accuracy*100:.2f}%  (on 5000-sample subset)")
print(f"CNN Accuracy           : {cnn_accuracy*100:.2f}%  (on full validation set)")
print("\n✅ Model Evaluation complete.")

### Confusion Matrices

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(22, 6))

cm_data_list = [
    (confusion_matrix(y_test_s, svm_pred),     "SVM Confusion Matrix",           "Blues"),
    (confusion_matrix(y_test_s, rf_pred),      "Random Forest Confusion Matrix", "Greens"),
    (confusion_matrix(y_true_cnn, y_pred_cnn), "CNN Confusion Matrix",           "Oranges"),
]

for ax, (cm_data, title, cmap) in zip(axes, cm_data_list):
    sns.heatmap(cm_data, annot=False, fmt='d', cmap=cmap, ax=ax,
                xticklabels=False, yticklabels=False)
    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")

plt.suptitle("Confusion Matrices — All Models", fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

### Classification Report — CNN

In [ ]:
target_names = [c.replace('___', ' — ').replace('_', ' ') for c in class_names]
print(classification_report(y_true_cnn, y_pred_cnn, target_names=target_names))